# Day 23: LangChain "Chains" vs "Expressions" (LCEL)

Welcome to Day 23! Today we are looking at **LangChain Expression Language (LCEL)**, the modern standard for building modular, composable, and production-ready RAG and agent pipelines.

As an AI Engineer, you'll find that legacy LangChain concepts (like `LLMChain` or `RetrievalQA`) often hide too much complexity, making them difficult to debug or extend. LCEL replaces these rigid structures with a functional approach. 

## The "Why" and "How" behind LCEL

**Why LCEL?**
1.  **Composability**: Instead of monolithic classes, LCEL lets you chain together modular components (Prompts, LLMs, Output Parsers, Retrievers) using the pipe operator `|`. 
2.  **Streaming & Async out of the box**: LCEL natively supports synchronous, asynchronous, streaming, and batched execution (`.invoke()`, `.stream()`, `.batch()`, `.ainvoke()`), which is critical for production web applications where latency matters.
3.  **Transparency**: It is much easier to see the data flow. You pass inputs, map variables, and pipe them directly into models and parsers.

**How it Works:**
At its core, LCEL relies on the `Runnable` protocol. Every element in an LCEL pipeline implements `Runnable`, meaning they all share the standard `.invoke()` and `.stream()` methods. When you use the pipe `|` (e.g., `prompt | model | parser`), you are creating a `RunnableSequence`.

### AI Security in LCEL
When building pipelines, always consider:
1. **PII Protection**: Sanitize inputs *before* passing them to external models.
2. **Prompt Injection**: Use proper prompt templates and validation to separate instructions from user input.
3. **Fallbacks**: LCEL allows defining fallback models using `.with_fallbacks()` so your app stays up if a provider goes down.


## 1. Code Implementation: The Basics vs LCEL

Let's look at how to implement a basic Q&A pipeline using LCEL. We will use `ChatGroq` or a mock model if no API key is available, but for strict adherence to non-fictitious, working code we'll use a `FakeListChatModel` from `langchain_core` to guarantee execution without API dependencies during testing. In production, this would be `ChatOpenAI`, `ChatAnthropic`, etc.


In [1]:
from typing import Dict, Any
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSerializable, RunnablePassthrough
# For testing/demonstration without an API key, we use a Fake model.
# In a real app, use: from langchain_openai import ChatOpenAI
from langchain_core.language_models.fake_chat_models import FakeListChatModel

def build_lcel_chain() -> RunnableSerializable:
    """
    Builds a simple LCEL chain that takes a topic and generates a joke.
    Returns:
        A compiled Runnable sequence.
    """
    # 1. Prompt Template
    prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}.")
    
    # 2. LLM (Using a fake model to ensure the code executes locally without keys)
    # The fake model simply cycles through the provided responses.
    model = FakeListChatModel(responses=["Why do Python programmers prefer dark mode? Because light attracts bugs!"])
    
    # 3. Output Parser (Extracts the string from the AIMessage object)
    parser = StrOutputParser()
    
    # 4. LCEL Pipeline composition using the | operator
    chain = prompt | model | parser
    
    return chain

# Execute the basic chain
print("--- Basic LCEL Execution ---")
chain = build_lcel_chain()
result = chain.invoke({"topic": "programming"})
print(f"Input: programming\nOutput: {result}\n")


--- Basic LCEL Execution ---
Input: programming
Output: Why do Python programmers prefer dark mode? Because light attracts bugs!



## 2. Medium: Clean OOP and State Management

Here we encapsulate the LCEL chain within a class. This allows us to manage state, inject dependencies (like the model), and provide a clean interface.


In [2]:
import re
from typing import Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSerializable
from langchain_core.language_models.fake_chat_models import FakeListChatModel

class JokeGeneratorPipeline:
    """
    An OOP wrapper for an LCEL joke generation pipeline.
    Manages its own state (e.g., generation count) and encapsulated chain logic.
    """
    def __init__(self, model: Optional[RunnableSerializable] = None):
        # Dependency Injection: Allow passing a specific model, or default to a safe FakeModel
        self.model = model or FakeListChatModel(responses=["Why did the AI cross the road? To optimize the objective function!"])
        self.chain = self._build_chain()
        self.generation_count = 0

    def _build_chain(self) -> RunnableSerializable:
        prompt = ChatPromptTemplate.from_template("Tell me a clever joke about {topic}.")
        return prompt | self.model | StrOutputParser()

    def generate(self, topic: str) -> str:
        """
        Executes the chain and updates internal state.
        """
        # Simple security: sanitize input to remove non-alphanumeric characters
        safe_topic = re.sub(r'[^a-zA-Z0-9 ]', '', topic)
        
        result = self.chain.invoke({"topic": safe_topic})
        self.generation_count += 1
        return result

# Execution
print("--- Medium OOP Execution ---")
pipeline = JokeGeneratorPipeline()
print(f"Joke: {pipeline.generate('machine learning')}")
print(f"Total jokes generated: {pipeline.generation_count}\n")


--- Medium OOP Execution ---
Joke: Why did the AI cross the road? To optimize the objective function!
Total jokes generated: 1



## 3. Advanced: Production RAG Pipeline with AI Security & Fallbacks

This example demonstrates a production-grade LCEL RAG implementation. It features strict type hinting, robust error handling, OOP principles, PII redaction, and model fallbacks (using `FakeListChatModel` as a safe fallback in this local environment).


In [3]:
import re
import os
import logging
from typing import List, Dict, Any, Optional
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableSerializable
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models.fake_chat_models import FakeListChatModel
# If using OpenAI in a real setting, use: from langchain_openai import ChatOpenAI

# Setup basic logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class FailingFakeChatModel(FakeListChatModel):
    """Custom fake model that explicitly raises an Exception to trigger fallbacks."""
    def _generate(self, messages: List[Any], stop: Optional[List[str]] = None, run_manager: Optional[Any] = None) -> Any:
        raise ValueError("Simulated Provider Outage - 500 Internal Server Error")

class ProductionRAGPipeline:
    """
    A robust RAG pipeline implementing LCEL, PII redaction, and model fallbacks.
    """
    def __init__(self):
        # In production, securely fetch keys. Do not hardcode fallbacks in getenv.
        self.api_key = os.environ.get("OPENAI_API_KEY")
        self.chain = self._build_secure_chain()

    def _mock_retriever(self, query: str) -> List[Document]:
        """Mock vector database retrieval."""
        return [
            Document(page_content="LCEL allows chaining components via the pipe operator."),
            Document(page_content="AI Security requires input sanitization before generation.")
        ]

    def _format_docs(self, docs: List[Document]) -> str:
        return "\n\n".join(doc.page_content for doc in docs)

    def _redact_pii(self, text: str) -> str:
        """
        Security best practice: Redact potential email addresses from input.
        """
        email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
        redacted = re.sub(email_pattern, '[REDACTED_EMAIL]', text)
        if redacted != text:
            logger.info("PII detected and redacted from input.")
        return redacted

    def _build_secure_chain(self) -> RunnableSerializable:
        prompt = ChatPromptTemplate.from_template(
            "Answer the question securely based on context:\nContext: {context}\nQuestion: {question}"
        )
        
        # Primary model (e.g., ChatOpenAI)
        primary_model = FailingFakeChatModel(responses=[])
        
        # Fallback model for high availability
        fallback_model = FakeListChatModel(responses=["Based on context, LCEL uses pipes and requires sanitization."])
        
        # Compose model with fallback
        robust_model = primary_model.with_fallbacks([fallback_model])
        
        setup_and_retrieval = RunnableParallel({
            "context": RunnableLambda(self._mock_retriever) | RunnableLambda(self._format_docs),
            # Apply PII redaction directly in the pipeline
            "question": RunnableLambda(self._redact_pii)
        })
        
        return setup_and_retrieval | prompt | robust_model | StrOutputParser()

    def query(self, user_question: str) -> str:
        """
        Executes the RAG pipeline with error handling.
        """
        try:
            return self.chain.invoke(user_question)
        except Exception as e:
            logger.error(f"Pipeline execution failed: {e}")
            return "I'm sorry, I cannot process your request at this time."

# Execution
print("--- Advanced Secure RAG Execution ---")
rag = ProductionRAGPipeline()
response = rag.query("How does LCEL work? Contact admin@company.com for details.")
print(f"Output: {response}\n")


INFO:__main__:PII detected and redacted from input.


--- Advanced Secure RAG Execution ---
Output: Based on context, LCEL uses pipes and requires sanitization.



## Common Pitfalls in Production

1. **Assuming APIs will never fail**: Network partitions happen. Always configure fallbacks and retries on your LCEL chains using `.with_fallbacks()`.
2. **Leaking PII**: Throwing unfiltered user queries at third-party models is a massive security risk. Always sanitize or redact data first.
3. **Hardcoding Secrets**: Never commit or hardcode fallback API keys. Fetch them securely from your environment variables at runtime.
4. **Over-complicating `RunnableMap` (RunnableParallel):** When dealing with complex inputs, developers often nest `RunnableParallel` too deeply. Break complex data preparation into standard Python functions and wrap them in a `@chain` decorator or a simple `RunnableLambda`.
5. **Forgetting the Output Parser:** If you don't end your chain with an `OutputParser` (like `StrOutputParser()`), `.invoke()` will return an `AIMessage` object instead of a clean string. This often breaks downstream API responses that expect JSON serialization.
6. **Debugging Black Boxes:** When a long pipe (`a | b | c | d`) fails, it can be hard to know which step broke. Use `.with_config({"run_name": "MyStep"})` on components, or attach LangSmith for tracing.


## Practical Lab: Translation and Formatting Chain

**Your Task:**
You need to build a two-step chain using LCEL.

1.  **Step 1 (Translation):** Take a user `input_text` and a `target_language`. Use a prompt to translate the text.
2.  **Step 2 (Formatting):** Take the output of the translation, and pass it to a second prompt that formats it as a formal email.
3.  **Requirement:** Chain these together using the `|` operator. 
4.  **Video Requirement:** Record a brief (2-3 min) async video walkthrough explaining your design decisions, specifically how data flows between the two chains.

*Hint: Use `RunnablePassthrough.assign()` or `RunnableParallel` to keep variables across the pipeline, or simply pipe the string output of the first chain into the input of the second prompt using another dict mapping.*


In [4]:
from langchain_core.runnables import RunnableLambda, RunnableSerializable
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models.fake_chat_models import FakeListChatModel

def build_translation_email_chain() -> RunnableSerializable:
    """
    Builds a two-stage LCEL pipeline: Translation -> Email Formatting.
    """
    # Stage 1: Translation
    translate_prompt = ChatPromptTemplate.from_template(
        "Translate the following text into {target_language}: {input_text}"
    )
    model = FakeListChatModel(responses=[
        "Bonjour le monde", # Response for stage 1
        "Subject: Formal Greeting\n\nBonjour le monde\n\nBest Regards." # Response for stage 2
    ])
    parser = StrOutputParser()
    
    translation_chain = translate_prompt | model | parser
    
    # Stage 2: Email Formatting
    email_prompt = ChatPromptTemplate.from_template(
        "Take the following text and format it as a formal business email: {translated_text}"
    )
    
    full_chain = (
        {"translated_text": translation_chain} 
        | email_prompt 
        | model 
        | parser
    )
    
    return full_chain

# Execute Lab task
print("\n--- Lab Execution: Translation & Email Chain ---")
lab_chain = build_translation_email_chain()
lab_result = lab_chain.invoke({
    "input_text": "Hello world", 
    "target_language": "French"
})
print("Final Output:")
print(lab_result)



--- Lab Execution: Translation & Email Chain ---
Final Output:
Subject: Formal Greeting

Bonjour le monde

Best Regards.


## Reference Links
- [LangChain Expression Language (LCEL) Documentation](https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language)
- [Runnable Interface Details](https://python.langchain.com/v0.2/docs/concepts/#runnable-interface)
- [Adding Fallbacks in LCEL](https://python.langchain.com/v0.2/docs/how_to/fallbacks/)
